# Phase 7 — Grounded RAG Pipeline (Colab)

Hybrid retrieval + Llama-3.1-8B-Instruct 4-bit grounded generator over
the Phase 3 IKS corpus (285 chunks: 78 Vrikshayurveda + 207 Brihat
Samhita 12-chapter subset).

## Design (master plan §17)

- **Retrieval:** dense (BAAI/bge-large-en-v1.5 over ChromaDB) +
  sparse (BM25) → Reciprocal-Rank-Fusion (k=60) → cross-encoder
  rerank (BAAI/bge-reranker-base). All three stages are toggleable
  for Phase 11 §27 ablations.
- **Generation:** Llama-3.1-8B-Instruct in 4-bit (nf4 + double-quant
  + bf16 compute). Master plan §17 grounded-advisor system prompt:
  answer ONLY from retrieved passages, cite source + chapter + verse,
  step-by-step organic protocol, refuse out-of-corpus questions.
- **Corpus transport:** chunks live in the private HF dataset
  `ankit-iiitdmj/iks-corpus-chunks` (the laptop's `corpus/vector_db/`
  cannot reach Colab). This notebook rebuilds ChromaDB in-session
  from the dataset.

## Platform

Phase 7 runs on Colab (Linux). On Windows, `chromadb` and `torch` /
`sentence_transformers` segfault in the same Python process — that's
captured in the memory entry `feedback-chromadb-torch-windows-dll`.
On Linux the single-process design is fine.

## ⚠️ Before you start

- **Runtime:** GPU (T4 free tier is enough for 4-bit 8B; expect
  ~5–6 GB VRAM).
- **HF Hub token:** Write token belonging to `ankit-iiitdmj` (needed
  for the private chunks dataset AND the gated Llama-3.1 weights).
- **Llama-3.1 license:** you must have accepted it at
  https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct before
  Cell 9 runs. If you haven't, Cell 9 will fail with a 403 on the
  weights download.


In [ ]:
# Cell 2 — setup: clone repo + install dependencies (defensive)
REPO_URL = "https://github.com/ankit8453/iks-rag-thesis.git"
REPO_PATH = "/content/iks-rag-thesis"

import os, subprocess, sys

if os.path.isdir(REPO_PATH) and not os.path.isfile(os.path.join(REPO_PATH, "requirements.txt")):
    print(f"Removing partial clone at {REPO_PATH} ...")
    subprocess.run(["rm", "-rf", REPO_PATH], check=True)
if not os.path.isdir(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)

os.chdir(REPO_PATH)
print("Repo root contents:", sorted(os.listdir(".")))

# Phase 7 runtime deps. Colab pre-installs torch / numpy / pandas /
# sklearn so we skip those. `bitsandbytes` is needed for 4-bit
# Llama; `rank-bm25` for sparse retrieval.
_pip_packages = [
    "transformers>=4.44",
    "bitsandbytes>=0.43",
    "accelerate>=0.33",
    "sentence-transformers>=3.0",
    "chromadb>=0.5",
    "rank-bm25>=0.2.2",
    "huggingface_hub>=0.24",
    "datasets>=2.20",
    "pyyaml>=6.0",
]
proc = subprocess.run(
    [sys.executable, "-m", "pip", "install", *_pip_packages],
    capture_output=True, text=True,
)
if proc.returncode != 0:
    print("PIP STDOUT (tail):\n" + proc.stdout[-3000:])
    print("PIP STDERR (tail):\n" + proc.stderr[-3000:])
    raise SystemExit("pip install failed — see tails above.")
print("setup ok")


In [ ]:
# Cell 3 — HF Hub login (private chunks dataset + gated Llama weights)
from huggingface_hub import login, HfApi
login()  # Colab inline widget — paste your Write token

_whoami = HfApi().whoami()
assert _whoami["name"] == "ankit-iiitdmj", (
    f"HF Hub token belongs to {_whoami['name']!r}, expected 'ankit-iiitdmj'."
)
print(f"HF Hub ok: user={_whoami['name']}")


In [ ]:
# Cell 4 — GPU check (4-bit 8B Llama needs ~5–6 GB; T4 is fine)
import subprocess, torch, sys
sys.path.insert(0, REPO_PATH)

subprocess.run(["nvidia-smi"], check=False)
print()
assert torch.cuda.is_available(), "No GPU detected — switch runtime to GPU before running Cell 9."
dev = torch.cuda.get_device_properties(0)
vram_gib = dev.total_memory / 1024**3
print(f"GPU: {dev.name}, VRAM: {vram_gib:.1f} GiB")
if vram_gib < 14:
    print("WARN: <14 GiB VRAM — Llama-3.1-8B 4-bit may OOM on contexts; consider "
          "`model_name=meta-llama/Llama-3.2-3B-Instruct` when constructing the generator.")


## Corpus rebuild

Pull the private 285-row chunks dataset from HF Hub and re-embed it
into a fresh ChromaDB collection at `corpus/vector_db/`. Re-running
the cell upserts by deterministic sha1 `chunk_id` — no duplicates.


In [ ]:
# Cell 6 — Load chunks from HF and rebuild ChromaDB locally
from src.rag.corpus_loader import load_chunks_from_hf, build_chroma

chunks = load_chunks_from_hf()  # ankit-iiitdmj/iks-corpus-chunks
print(f"Loaded {len(chunks)} chunks; first chunk:")
print(" ", {k: chunks[0][k] for k in ("book_id", "chapter", "verse_or_section", "chunk_id")})

collection = build_chroma(chunks, persist_dir="corpus/vector_db")
print(f"\nCollection populated: count={collection.count()}")
assert collection.count() == len(chunks), "Chroma count mismatch — re-run Cell 6."


In [ ]:
# Cell 7 — Build HybridRetriever (dense + sparse + reranker, all on)
from src.rag.retriever import HybridRetriever

retriever = HybridRetriever(collection, use_dense=True, use_sparse=True, use_reranker=True)
print(
    f"HybridRetriever ready: dense={retriever.use_dense} "
    f"sparse={retriever.use_sparse} reranker={retriever.use_reranker}"
)


In [ ]:
# Cell 8 — Retriever smoke. Runs BEFORE Cell 9 so a retrieval failure
# fails fast (no need to pay the ~5 GB Llama download cost).
SMOKE_QUERIES = [
    "how to treat a diseased tree",                    # Vrikshayurveda / Brihat ch.55
    "signs that predict rainfall",                     # Brihat ch.21-28
    "how to find underground water",                   # Brihat ch.54
    "yellow leaf disease and the correct soil for it", # joint disease + soil (Phase 8 preview)
    "organic protocol for sandy loam crops",          # cross-source retrieval
]

for q in SMOKE_QUERIES:
    hits = retriever.retrieve(q, k=5)
    print("=" * 78)
    print(f"QUERY: {q!r}")
    for i, h in enumerate(hits, 1):
        meta = h.metadata or {}
        src = f"{meta.get('source_text','?')} ch.{meta.get('chapter','?')} v.{meta.get('verse_or_section','?')}"
        snip = (h.text or '').replace('\n', ' ')[:140]
        print(f"  [{i}] score={h.score:.4f} stage={h.retriever}  {src}")
        print(f"        {snip}")
    print()


In [ ]:
# Cell 9 — Load Llama-3.1-8B 4-bit (gated — Cell 3's token must have license access)
import torch
from src.rag.generator import GroundedGenerator

generator = GroundedGenerator(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    load_in_4bit=True,
    temperature=0.2,
    max_new_tokens=512,
    seed=42,
)
generator._ensure_loaded()  # noqa: SLF001 — warm up here so VRAM is visible BEFORE Cell 10
torch.cuda.empty_cache()
mem = torch.cuda.memory_allocated() / 1024**3
print(f"Llama-3.1-8B 4-bit loaded. CUDA memory in use: {mem:.2f} GiB")


In [ ]:
# Cell 10 — End-to-end RAG: retriever → grounded generator, 5 demo queries
from src.rag.pipeline import RAGPipeline

pipeline = RAGPipeline(retriever=retriever, generator=generator, default_k=5)

DEMO_QUERIES = [
    # In-corpus expected to work well:
    "How should a diseased tree with falling branches be treated?",
    "What signs in the sky predict imminent rainfall?",
    "How does the classical text guide finding underground water?",
    # Joint disease + soil — foreshadows Phase 8 (multimodal context):
    "What organic protocol should be used for a tree showing yellow leaves growing in mixed sandy-loam soil?",
    # OUT-OF-CORPUS faithfulness check: the model MUST refuse rather than hallucinate.
    "What is the recommended drone-spraying schedule for monoculture rice fields?",
]

for q in DEMO_QUERIES:
    print("=" * 78)
    print(f"QUERY: {q}")
    result = pipeline.answer(q, k=5)
    print("--- ANSWER ---")
    print(result.answer)
    print("--- CITATIONS ---")
    for c in result.citations:
        print("  -", c)
    print("--- CHUNKS USED ---")
    for cid in result.used_chunk_ids:
        # show the metadata of every chunk the model actually cited
        match = next((r for r in result.retrieved if r.chunk_id == cid), None)
        if match is not None:
            meta = match.metadata
            print(f"  {cid[:10]}  {meta.get('source_text')} ch.{meta.get('chapter')} v.{meta.get('verse_or_section')}")
    print("--- TOP RETRIEVED (for inspection) ---")
    for i, h in enumerate(result.retrieved[:3], 1):
        meta = h.metadata
        src = f"{meta.get('source_text','?')} ch.{meta.get('chapter','?')} v.{meta.get('verse_or_section','?')}"
        snip = (h.text or '').replace('\n',' ')[:120]
        print(f"  [{i}] score={h.score:.4f} {src} :: {snip}")
    print()


## How to read these results

- **In-corpus queries 1–3** should each produce a numbered organic
  protocol with one or more `[Source Text, ch.X, v.Y]` citations
  matching the top retrieved chunks. The `CHUNKS USED` row tells you
  which retrieved chunks the answer actually cited.
- **Joint query 4** is a preview of Phase 8 (multimodal context):
  the question deliberately mentions both a disease symptom and a
  soil type, so retrieval should pull chunks from both Vrikshayurveda
  (disease) and Brihat Samhita (soil/exploration). The Phase 8
  notebook will inject vision-module predictions into the query at
  exactly this seam.
- **Out-of-corpus query 5** is the §17 faithfulness sanity check.
  The model MUST emit the locked refusal sentence — *"The retrieved
  classical-text passages do not contain enough information to
  answer this question. Please consult a qualified agricultural
  expert."* — rather than hallucinating a drone-spraying schedule.
  If it hallucinates a treatment instead, log it as a Phase 11 RAGAS
  faithfulness issue and do NOT hide it.


## Phase 7 complete

Pipeline lives at:

- `src/rag/corpus_loader.py` (`load_chunks_from_hf`, `build_chroma`)
- `src/rag/retriever.py` (`HybridRetriever`, `RetrievedChunk`)
- `src/rag/generator.py` (`GroundedGenerator`, §17 prompt)
- `src/rag/pipeline.py` (`RAGPipeline`)

### Next: Phase 8 (multimodal integration)

The query-construction step is the seam — Phase 8 will compose the
user question with the Phase 5 disease classifier's prediction and
the Phase 6 soil classifier's prediction (e.g. *"yellow leaves on
Bottle Gourd in sandy-loam soil"*) and route the enriched query
through this same `RAGPipeline`.

### Swapping LLMs

If VRAM is tight (T4 free tier with long contexts), pass
`model_name="meta-llama/Llama-3.2-3B-Instruct"` to either
`GroundedGenerator(...)` or `RAGPipeline(model_name=...)`. The pipeline
is generator-agnostic, so the rest of the code is unchanged.

### Adding more books

Once Krishi Parashara, Upavanavinoda, Kashyapiyakrishisukti, or the
TBD-sixth text are processed by Phase 3, re-run
`python scripts/push_corpus_chunks.py` on the laptop to push the
expanded corpus to `ankit-iiitdmj/iks-corpus-chunks`. The next Colab
Cell 6 run picks them up automatically — no code change anywhere in
the RAG pipeline.
